# Verify Dynamic Center of Pressure (EQ-COP-*)

Strip-theory CoP from precomputed Cp archives + SciPy chordwise/spanwise integration.

**Governing equations**
- EQ-COP-001: $c_n = \int_0^1 \Delta C_p\,d(x/c)$
- EQ-COP-002: $x_{cp}/c = \int_0^1 (x/c)\Delta C_p\,d(x/c) / c_n$
- EQ-COP-003: $dL = q\,c(z)\,c_n(z)\,dz$
- EQ-COP-004: $z_{cp} = \int z\,dL / L$
- EQ-COP-007: $\delta_{req} = C_{L,req}/C_{L\alpha,3D}$ (Helmbold; $\alpha=\delta$)

Shaft/hinge remains at 25% chord; CoP is free.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from auv_fin_design.domain.center_of_pressure import CoPSolverConfig, solve_center_of_pressure
from auv_fin_design.domain.center_of_pressure.xfoil_provider import XfoilProvider
from auv_fin_design.domain.geometry.sizing import build_fin_from_planform
from auv_fin_design.infrastructure.config.loader import repo_root

print('repo', repo_root())
print('airfoils', list((repo_root()/'data'/'airfoils').iterdir()))

## Cp(x) at design alpha

In [ ]:
prov = XfoilProvider()
pd = prov.load_pressure_distribution('naca0015', 2e5, 5.0)
x = np.array(pd.x_c); dcp = np.array(pd.dcp)
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(x, pd.cp_upper, label='Cp upper')
ax.plot(x, pd.cp_lower, label='Cp lower')
ax.plot(x, dcp, label='ΔCp', lw=2)
ax.invert_yaxis(); ax.set_xlabel('x/c'); ax.set_ylabel('Cp'); ax.legend(); ax.set_title('NACA0015 α=5°')
plt.show()

## Strip discretization and loads

In [ ]:
g = build_fin_from_planform(root_chord=0.12, tip_chord=0.06, span=0.09, thickness_ratio=0.15)
r = solve_center_of_pressure(
    g, airfoil='naca0015', dynamic_pressure=1123.0, speed_mps=1.5, nu=1.004e-6,
    alpha_deg=5.0, cm_polar=0.0, cl_polar=0.4,
    config=CoPSolverConfig(n_strips=40),
    lift_required_n=2.0, cl_alpha_2d_per_rad=2*np.pi, stall_alpha_deg=12.0, stall_margin_deg=5.0,
)
z = [s.z_m for s in r.strips]; L = [s.lift_n for s in r.strips]; xc = [s.cp_x_frac for s in r.strips]
fig, axs = plt.subplots(1,3, figsize=(12,3.5))
axs[0].plot(z, L, 'o-'); axs[0].set_title('Strip lift'); axs[0].set_xlabel('z [m]')
axs[1].plot(z, xc, 'o-'); axs[1].axhline(0.25, ls='--', color='k'); axs[1].set_title('Strip x_cp/c')
axs[2].bar(['QC','Cm/CL','Integrated'], [0.25, r.verification.x_cp_c_cm_cl, r.x_cp_le_frac])
axs[2].set_title('x_cp/c comparison')
plt.tight_layout(); plt.show()
print(r.verification.message)
print('3D CoP mm:', r.x_cp_from_le_m*1000, r.y_cp_m*1000, r.z_cp_m*1000)
if r.deflection:
    print('delta_req', r.deflection.delta_required_deg, 'usable', r.deflection.delta_max_usable_deg)

## CoP migration with alpha and Re

In [ ]:
alphas = np.linspace(1, 12, 12)
xc_a, zc_a = [], []
for a in alphas:
    rr = solve_center_of_pressure(
        g, airfoil='naca0015', dynamic_pressure=1123.0, speed_mps=1.5, nu=1.004e-6,
        alpha_deg=float(a), cm_polar=0.0, cl_polar=0.4, config=CoPSolverConfig(n_strips=20),
    )
    xc_a.append(rr.x_cp_le_frac); zc_a.append(rr.z_cp_m)
fig, ax = plt.subplots(1,2, figsize=(10,3.5))
ax[0].plot(alphas, xc_a); ax[0].axhline(0.25, ls='--'); ax[0].set_xlabel('α [deg]'); ax[0].set_ylabel('x_cp/c')
ax[1].plot(alphas, np.array(zc_a)*1000); ax[1].set_xlabel('α [deg]'); ax[1].set_ylabel('z_cp [mm]')
plt.tight_layout(); plt.show()

speeds = [0.8, 1.2, 1.5, 2.0]
xc_r = []
for V in speeds:
    q = 0.5*998.2*V*V
    rr = solve_center_of_pressure(
        g, airfoil='naca0015', dynamic_pressure=q, speed_mps=V, nu=1.004e-6,
        alpha_deg=5.0, cm_polar=0.0, cl_polar=0.4, config=CoPSolverConfig(n_strips=20),
    )
    xc_r.append(rr.x_cp_le_frac)
plt.figure(figsize=(5,3)); plt.plot(speeds, xc_r, 'o-'); plt.xlabel('V [m/s]'); plt.ylabel('x_cp/c'); plt.title('Re sensitivity via speed'); plt.show()

## Aspect-ratio sensitivity (same area)

In [ ]:
S = 0.008
ars = [0.8, 1.2, 1.8, 2.5]
zc = []
for ar in ars:
    b = np.sqrt(S*ar); cr = 2*S/(b*1.5); ct = 0.5*cr
    gg = build_fin_from_planform(root_chord=cr, tip_chord=ct, span=b, thickness_ratio=0.15)
    rr = solve_center_of_pressure(
        gg, airfoil='naca0015', dynamic_pressure=1123.0, speed_mps=1.5, nu=1.004e-6,
        alpha_deg=5.0, cm_polar=0.0, cl_polar=0.4, config=CoPSolverConfig(n_strips=20),
    )
    zc.append(rr.z_cp_m/b)
plt.figure(figsize=(5,3)); plt.plot(ars, zc, 'o-'); plt.xlabel('AR'); plt.ylabel('z_cp / span'); plt.title('AR sensitivity'); plt.show()
print('OK')